# ☁️ Upload OHLC HDF5 to Cloudflare R2

Uploads the local `stocks_data_latest.h5` panel to `r2://<bucket>/data/stocks_data_latest.h5`.

**Required env vars**
| Variable | Description |
|----------|-------------|
| `R2_ACCESS_KEY_ID` | R2 API token key ID |
| `R2_SECRET_ACCESS_KEY` | R2 API token secret |
| `R2_ENDPOINT_URL` | _(optional)_ defaults to the account endpoint below |
| `R2_BUCKET` | _(optional)_ defaults to `all-in-one-porffolio` |
| `R2_OHLC_KEY` | _(optional)_ defaults to `data/stocks_data_latest.h5` |
| `R2_MODELS_PREFIX` | _(optional)_ defaults to `models/` — meta-label artifacts on R2 (see §6) |

Section **6** syncs `r2://<bucket>/models/` → **`backend/models/`** (local).

# 1. Environment Setup

In [2]:
import os
import sys
from pathlib import Path

import boto3
from botocore.exceptions import ClientError
from dotenv import load_dotenv
from tqdm.notebook import tqdm

load_dotenv()

NOTEBOOKS_DIR = Path(".").resolve()
PROJECT_ROOT  = NOTEBOOKS_DIR.parent

# 2. Configuration

In [4]:
# ── Local source ─────────────────────────────────────────────────────────────
LOCAL_H5 = Path(
    os.getenv("LOCAL_OHLC_H5", str(NOTEBOOKS_DIR / "stocks_data_latest.h5"))
)

# ── R2 destination ────────────────────────────────────────────────────────────
R2_ENDPOINT   = os.getenv("R2_ENDPOINT_URL", "https://bf992ebd6b2d460f07db4868252c33a6.r2.cloudflarestorage.com")
R2_BUCKET     = os.getenv("R2_BUCKET", "all-in-one-porffolio")
R2_OHLC_KEY   = os.getenv("R2_OHLC_KEY", "data/stocks_data_latest.h5")

print(f"Source : {LOCAL_H5}  ({LOCAL_H5.stat().st_size / 1024**2:.1f} MB)")
print(f"Target : r2://{R2_BUCKET}/{R2_OHLC_KEY}")
print(f"Endpoint: {R2_ENDPOINT}")

Source : /Users/phuchuynh/Work/all-in-one-portfolio/notebooks/stocks_data_latest.h5  (18.8 MB)
Target : r2://all-in-one-porffolio/data/stocks_data_latest.h5
Endpoint: https://bf992ebd6b2d460f07db4868252c33a6.r2.cloudflarestorage.com


# 3. Verify HDF5 Contents

In [ ]:
import pandas as pd

with pd.HDFStore(LOCAL_H5, mode="r") as store:
    keys = store.keys()
    print("Keys:", keys)
    panel = store["/stocks"]

symbols = panel.columns.get_level_values("symbol").unique().tolist()
print(f"Shape  : {panel.shape}")
print(f"Date range: {panel.index.min()} → {panel.index.max()}")
print(f"Symbols ({len(symbols)}): {symbols[:10]} ...")

# 4. Upload to R2

In [ ]:
s3 = boto3.client(
    "s3",
    endpoint_url=R2_ENDPOINT,
    aws_access_key_id=os.getenv("R2_ACCESS_KEY_ID", "5f87afb6b4f0890096384f3e75d6d569"),
    aws_secret_access_key=os.getenv("R2_SECRET_ACCESS_KEY", "aca9418d6846c29215c76ff4dbc742d6af76a14308509219daf1a72e4b87b5ee"),
    region_name="auto",
)

In [ ]:
file_size = LOCAL_H5.stat().st_size

class _ProgressBar:
    """Callback for boto3 multipart upload progress."""
    def __init__(self, total: int):
        self._bar = tqdm(total=total, unit="B", unit_scale=True, desc="Uploading")

    def __call__(self, bytes_transferred: int):
        self._bar.update(bytes_transferred)

    def close(self):
        self._bar.close()

progress = _ProgressBar(file_size)
try:
    s3.upload_file(
        str(LOCAL_H5),
        R2_BUCKET,
        R2_OHLC_KEY,
        Callback=progress,
    )
finally:
    progress.close()

print(f"\n✓ Upload complete → r2://{R2_BUCKET}/{R2_OHLC_KEY}")

# 5. Verify Upload

In [ ]:
resp = s3.head_object(Bucket=R2_BUCKET, Key=R2_OHLC_KEY)
remote_size = resp["ContentLength"]
last_modified = resp["LastModified"]

print(f"Remote size   : {remote_size / 1024**2:.1f} MB")
print(f"Local size    : {file_size / 1024**2:.1f} MB")
print(f"Last modified : {last_modified}")

assert remote_size == file_size, f"Size mismatch: remote={remote_size} local={file_size}"
print("✓ Size check passed")

# 6. Sync models from R2 → `backend/models/`

Downloads every object under `r2://<bucket>/<R2_MODELS_PREFIX>` into **`backend/models/`**, preserving subpaths. This matches the prefix used when `train_meta_label_models_standalone.py` uploads artifacts (`R2_MODELS_PREFIX`, default `models/`).

**Prerequisites:** §1–2 (imports + `load_dotenv`). Set `R2_ACCESS_KEY_ID` and `R2_SECRET_ACCESS_KEY` (e.g. in `notebooks/.env`).

| Variable | Description |
|----------|-------------|
| `R2_MODELS_PREFIX` | _(optional)_ defaults to `models/` |
| `LOCAL_MODELS_DIR` | _(optional)_ defaults to `<project>/backend/models` |

In [6]:
R2_SYNC_ENDPOINT = os.getenv(
    "R2_ENDPOINT_URL",
    "https://bf992ebd6b2d460f07db4868252c33a6.r2.cloudflarestorage.com",
)
R2_SYNC_BUCKET = os.getenv("R2_BUCKET", "all-in-one-porffolio")
R2_MODELS_PREFIX = os.getenv("R2_MODELS_PREFIX", "models/").strip()
if R2_MODELS_PREFIX.startswith("/"):
    R2_MODELS_PREFIX = R2_MODELS_PREFIX[1:]
if R2_MODELS_PREFIX and not R2_MODELS_PREFIX.endswith("/"):
    R2_MODELS_PREFIX += "/"

LOCAL_MODELS_DIR = Path(
    os.getenv("LOCAL_MODELS_DIR", str(PROJECT_ROOT / "backend" / "models"))
)
LOCAL_MODELS_DIR.mkdir(parents=True, exist_ok=True)

_key = os.getenv("R2_ACCESS_KEY_ID", "5f87afb6b4f0890096384f3e75d6d569")
_secret = os.getenv("R2_SECRET_ACCESS_KEY", "aca9418d6846c29215c76ff4dbc742d6af76a14308509219daf1a72e4b87b5ee")
if not _key or not _secret:
    raise RuntimeError("Set R2_ACCESS_KEY_ID and R2_SECRET_ACCESS_KEY (e.g. in notebooks/.env)")

s3_sync = boto3.client(
    "s3",
    endpoint_url=R2_SYNC_ENDPOINT,
    aws_access_key_id=_key,
    aws_secret_access_key=_secret,
    region_name="auto",
)

print(f"Source : r2://{R2_SYNC_BUCKET}/{R2_MODELS_PREFIX}*")
print(f"Target : {LOCAL_MODELS_DIR}")

keys_to_fetch: list[str] = []
paginator = s3_sync.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket=R2_SYNC_BUCKET, Prefix=R2_MODELS_PREFIX):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        if key.endswith("/"):
            continue
        keys_to_fetch.append(key)

if not keys_to_fetch:
    print("No objects under that prefix — nothing to download.")
else:
    for key in tqdm(keys_to_fetch, desc="Downloading models"):
        rel = key[len(R2_MODELS_PREFIX) :] if key.startswith(R2_MODELS_PREFIX) else key
        if not rel:
            continue
        dest = LOCAL_MODELS_DIR / rel
        dest.parent.mkdir(parents=True, exist_ok=True)
        s3_sync.download_file(R2_SYNC_BUCKET, key, str(dest))

    print(f"\n✓ Synced {len(keys_to_fetch)} file(s) → {LOCAL_MODELS_DIR}")

Source : r2://all-in-one-porffolio/models/*
Target : /Users/phuchuynh/Work/all-in-one-portfolio/backend/models



✓ Synced 5 file(s) → /Users/phuchuynh/Work/all-in-one-portfolio/backend/models
